In [8]:
# test similarity functions
from sklearn.metrics.pairwise import cosine_similarity
import torch
import torch.nn.functional as F

def cos_sklearn(x,y):
    return cosine_similarity(x,y)

def cos_torch(x,y):
    return F.cosine_similarity(x,y)

def cos_mul(x,y):
    x_norm = F.normalize(x, p=2, dim=1)
    y_norm = F.normalize(y, p=2, dim=1)
    
    return torch.matmul(x_norm, y_norm.T)

x = torch.randn(3, 2)
y = torch.randn(3, 2)

# counting the time
import time
start = time.time()
output_sklearn = cos_sklearn(x,y)
print("sklearn time:", time.time()-start)
start = time.time()
output_torch = cos_torch(x,y)
print("torch time:", time.time()-start)

start = time.time()
output_mul = cos_mul(x,y)
print("mul time:", time.time()-start)

#check if the results are the same
#if not torch.allclose(torch.tensor(output_sklearn), output_mul):
print(output_sklearn)
print(output_mul)



sklearn time: 0.0004911422729492188
torch time: 0.0003008842468261719
mul time: 0.00010538101196289062
[[-0.51267314  0.9996098  -0.36333653]
 [ 0.64820284 -0.9810736   0.5130834 ]
 [-0.6704705   0.9749085  -0.5382793 ]]
tensor([[-0.5127,  0.9996, -0.3633],
        [ 0.6482, -0.9811,  0.5131],
        [-0.6705,  0.9749, -0.5383]])


In [12]:
import torch

# Example list of tensors
votes = x

# Convert each tensor to a tuple for hashing/comparison
votes_as_tuples = [tuple(vote) for vote in votes]

# Custom counting function with .all() for complete equality
def count_occurrences(target, items):
    # Convert target back to a tensor for `.all()`
    target_tensor = torch.tensor(target)
    return sum((torch.tensor(item) == target_tensor).all().item() for item in items)

# Apply max with the custom count function
majority_vote_tuple = max(set(votes_as_tuples), key=lambda x: count_occurrences(x, votes_as_tuples))

# Convert majority vote back to a tensor if needed
majority_vote_tensor = torch.tensor(majority_vote_tuple)

print("Majority vote:", majority_vote_tensor)


Majority vote: tensor([0, 0, 1])


In [41]:
def is_nested(lst):
    if all(t.numel() == 1 for t in lst):  # Check if all tensors have 1 element (like in y)
        return any(isinstance(i, list) for i in lst)
    elif all(t.dim() == 1 and t.numel() > 1 for t in lst):  # Check if all tensors have more than 1 element and 1 dimension (like in x)
        return True
    

x = [torch.tensor([0,0,0]),torch.tensor([0,0,1]),torch.tensor([0,0,1]),torch.tensor([0,0,1])]
y = [torch.tensor(0),torch.tensor(0),torch.tensor(0),torch.tensor(1),torch.tensor(0),torch.tensor(0),torch.tensor(1),torch.tensor(1),torch.tensor(1),torch.tensor(0),torch.tensor(0)]
print(is_nested(x))

if is_nested(x):
    votes = [tuple(tensor.tolist()) for tensor in x[:3]]
    majority_vote = torch.tensor(max(set(votes), key=votes.count))
else:
    votes = x[:3]
    majority_vote = max(set(votes), key=votes.count)

print(majority_vote)

True
tensor([0, 0, 1])


# Imports

In [1]:
import sys
sys.path[0] = "/home/ge26xaj/./projects/fm_histopathology/scripts/"
print("In module products sys.path[0], __package__ ==", sys.path[0], __package__)
import warnings
warnings.filterwarnings("ignore")

import os
import ast
# from skimage import io
import argparse
import datetime
import numpy as np
import pandas as pd
import time
import torch
from torchvision.models.feature_extraction import create_feature_extractor
import torchvision
# import torch.backends.cudnn as cudnn
import json
import uuid
import math
import h5py
from sklearn.decomposition import PCA

# from monai.transforms import Compose, EnsureChannelFirst, RandRotate90, ResizeWithPadOrCrop, ScaleIntensity, RandFlip, RandAffine, BorderPad, Resize, adaptor, ScaleIntensityRange

from pathlib import Path

from timm.data import Mixup
from timm.models import create_model
from timm.loss import LabelSmoothingCrossEntropy, SoftTargetCrossEntropy
from timm.scheduler import create_scheduler
from timm.optim import create_optimizer
from timm.utils import NativeScaler, get_state_dict, ModelEma

# from models.deit.datasets import build_dataset
# from models.deit.engine import train_one_epoch, evaluate
# from models.deit.losses import DistillationLoss
# from models.deit.samplers import RASampler
# from models.deit.augment import new_data_aug_generator

# import models.deit.models as models
# import models.deit.models_v2 as models_v2

# import models.deit.utils as utils

from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score
from sklearn.metrics.pairwise import cosine_similarity

import os
# from bitarray import util as butil

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

In module products sys.path[0], __package__ == /home/ge26xaj/./projects/fm_histopathology/scripts/ None


In [2]:
import sys
sys.path[0] = "/home/ge26xaj/./projects/fm_histopathology/scripts/"
print("In module products sys.path[0], __package__ ==", sys.path[0], __package__)
import warnings
warnings.filterwarnings("ignore")

import os
import ast
from skimage import io
import argparse
import datetime
import numpy as np
import pandas as pd
import time
import torch
from torchvision.models.feature_extraction import create_feature_extractor
import torchvision
# import torch.backends.cudnn as cudnn
import json
import uuid
import math

from monai.transforms import Compose, EnsureChannelFirst, RandRotate90, ResizeWithPadOrCrop, ScaleIntensity, RandFlip, RandAffine, BorderPad, Resize, adaptor, ScaleIntensityRange

from pathlib import Path

from timm.data import Mixup
from timm.models import create_model
from timm.loss import LabelSmoothingCrossEntropy, SoftTargetCrossEntropy
from timm.scheduler import create_scheduler
from timm.optim import create_optimizer
from timm.utils import NativeScaler, get_state_dict, ModelEma

from models.deit.datasets import build_dataset
from models.deit.engine import train_one_epoch, evaluate
from models.deit.losses import DistillationLoss
from models.deit.samplers import RASampler
from models.deit.augment import new_data_aug_generator

import models.deit.models as models
import models.deit.models_v2 as models_v2

import models.deit.utils as utils

from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score
from sklearn.metrics.pairwise import cosine_similarity

import bitarray, os
from bitarray import util as butil

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

In module products sys.path[0], __package__ == /home/ge26xaj/./projects/fm_histopathology/scripts/ None


# Paths

In [6]:
models_root_path = "../models/"
# model_name = "deit_tiny_patch16_224"
# model_name = "general_pretrained_deit_tiny_patch16_224"
model_name = "camelyon_pca_pretrained_mocov3_tiny_checkpoint"

data_root_path = "../../../../../mnt/data/"

# Functions

In [7]:
from timm.models.vision_transformer import VisionTransformer, _cfg
from timm.models.registry import register_model
from timm.models.layers import trunc_normal_
import torch
import torch.nn as nn
from functools import partial


def deit_tiny_patch16_224(pretrained=False, **kwargs):
    model = VisionTransformer(
        patch_size=16, embed_dim=192, depth=12, num_heads=3, mlp_ratio=4, qkv_bias=True,
        norm_layer=partial(nn.LayerNorm, eps=1e-6), **kwargs)
    model.default_cfg = _cfg()
    if pretrained:
        checkpoint = torch.hub.load_state_dict_from_url(
            url="https://dl.fbaipublicfiles.com/deit/deit_tiny_patch16_224-a1311bcf.pth",
            map_location="cpu", check_hash=True
        )
        model.load_state_dict(checkpoint["model"])
    return model

In [8]:
def get_search_image_index(data, search_id, nested=True):
    search_id = str(search_id)
    for i, item in enumerate(data):
        if nested:
            if str(item[0]) == search_id:
                return i
        else:
            if str(item) == search_id:
                return i

@torch.no_grad()
def similarity_test(model_name, k, image_id, embedds=None, scratch=None, data=None):
    translate = {"deit_small_patch16_224":"pretrained_mocov3_tiny_checkpoint", "deit_tiny_patch16_224":"deit_tiny_patch16_224", "camelyon_UNI":"camelyon_UNI"}
    similarities = []
    embeddings = []
    return_embeddings = []
    all_ids = []
    all_targets = []

    if not embedds:
        if scratch:
            checkpoint = torch.load(os.path.join(models_root_path, f"{data}_{translate[model_name]}", "best_checkpoint.pth"), map_location='cpu')
            args = checkpoint["args"]
            device = "cpu"

            model = create_model(
                model_name,
                pretrained=False,
                num_classes=args.nb_classes, #1000 for pretrained
                drop_rate=0.0,
                drop_path_rate=0.1,
                drop_block_rate=None,
                img_size=224
            )
            if scratch.startswith('https'):
                checkpoint = torch.hub.load_state_dict_from_url(
                    scratch, map_location='cpu', check_hash=True)
            # else:
                # checkpoint = torch.load(scratch, map_location='cpu')
                if not model == "deit_small_patch16_224":
                    del checkpoint["model"]['head.weight']
                    del checkpoint["model"]['head.bias']
        else:
            checkpoint = torch.load(os.path.join(models_root_path, model_name, "best_checkpoint.pth"), map_location='cpu')
            args = checkpoint["args"]
            device = "cpu"

            model = create_model(
                args.model,
                pretrained=False,
                num_classes=args.nb_classes, #1000 for pretrained
                drop_rate=args.drop,
                drop_path_rate=args.drop_path,
                drop_block_rate=None,
                img_size=args.input_size
            )
            # model.load_state_dict(checkpoint['model_ema'])
            model.load_state_dict(checkpoint['model'])

        dataset_val, similarity_dataset, _ = build_dataset(is_train=False, is_test=True, args=args)
        sampler_val = torch.utils.data.SequentialSampler(dataset_val)
        data_loader = torch.utils.data.DataLoader(
                dataset_val, sampler=sampler_val,
                batch_size=int(1.5 * args.batch_size),
                num_workers=args.num_workers,
                pin_memory=args.pin_mem,
                drop_last=False
            )
        if args.bce_loss:
            criterion = torch.nn.BCEWithLogitsLoss()
        else:
            criterion = torch.nn.CrossEntropyLoss()

        metric_logger = utils.MetricLogger(delimiter="  ")
        header = 'Similarity Test:'

        # switch to evaluation mode
        model.eval()

        for batch in metric_logger.log_every(data_loader, 10, header):
            if args.data_set == 'WSS':
                ids = batch["id"]
                images = batch["image"]
                target = batch["label"]

            if args.data_set == 'patch_cam' or args.data_set == 'mhist' or args.data_set == 'crc':
                images, target = batch
                ids = target

            images = images.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            activation = {}
            def get_activation(name):
                def hook(model, input, output):
                    activation[name] = output.detach()
                return hook

            # compute output
            with torch.cuda.amp.autocast():
                output = model(images)
                if args.data_set == 'crc':
                    output = output
                else:
                    output = output.squeeze(1) #not in crc
                loss = criterion(output, target)

                model.pre_logits.register_forward_hook(get_activation("embed"))
                model(images)

            embeddings.append(activation['embed'])
            all_ids.extend(ids)
            all_targets.extend(target)
            # f1, acc = accuracy(output, target, topk=(1, 5))
            if args.data_set == 'crc':
                pred = output.softmax(dim=1)
                _, pred = torch.max(pred, 1)
                pred = np.array(pred.detach().cpu(), dtype=int) #softmax!
            else:
                pred = output.sigmoid()
                pred = np.array(output.detach().cpu() > 0.5, dtype=float)

            if args.data_set == 'WSS':
                f1 = f1_score(target.detach().cpu(), pred, average='samples')
            if args.data_set == 'crc':
                f1 = f1_score(y_true=np.array(target.detach().cpu()), y_pred=pred, average='weighted')
            if args.data_set == 'patch_cam' or args.data_set == 'mhist':
                f1 = f1_score(target.detach().cpu(), pred, average='binary')
            acc = accuracy_score(target.detach().cpu(), pred)
            if args.data_set == 'patch_cam' or args.data_set == 'mhist' or args.data_set == 'crc':
                bacc = balanced_accuracy_score(target.detach().cpu(), pred)

            batch_size = images.shape[0]
            metric_logger.update(loss=loss.item())
            metric_logger.meters['f1'].update(f1.item(), n=batch_size)
            metric_logger.meters['acc'].update(acc.item(), n=batch_size)
            if args.data_set == 'patch_cam' or args.data_set == 'mhist' or args.data_set == 'crc':
                metric_logger.meters['bacc'].update(bacc.item(), n=batch_size)

        if args.data_set == 'WSS':
            search_index = get_search_image_index(all_ids, image_id, False)
        if args.data_set == 'patch_cam' or args.data_set == 'mhist' or args.data_set == 'crc':
            search_index = image_id
    else:
        if model_name[:8] == "camelyon":
            all_ids = embedds[0]
            all_targets = embedds[1]
            embeddings = embedds[2]
        else:
            all_ids = [row[0] for row in embedds]
            all_targets = [row[1] for row in embedds]
            embeddings = [row[2] for row in embedds]
        search_index = image_id

    if not model_name[:8] == "camelyon": 
        embeddings = torch.cat(embeddings, dim=0)
    # print(embeddings.shape)
    for i, _ in enumerate(embeddings):
        similarity = cosine_similarity(embeddings[search_index].reshape(1, -1), embeddings[i].reshape(1, -1))
        return_embeddings.append([all_ids[i], all_targets[i], embeddings[i].reshape(1, -1)])
        similarities.append([all_ids[i], all_targets[i], similarity])
        # print(f"Similarity between {ids[0]}_{target[0].detach().cpu()}-{ids[i]}_{target[i].detach().cpu()}: {similarity}")
    similarities.sort(key = lambda row: row[2], reverse=True)
    
    if not embedds:
        if args.data_set == 'patch_cam' or args.data_set == 'mhist' or args.data_set == 'crc':
            print('* F1 {top1.global_avg:.4f} ACC {top5.global_avg:.4f} BACC {top6.global_avg:.4f} loss {losses.global_avg:.4f}'
                .format(top1=metric_logger.f1, top5=metric_logger.acc, top6=metric_logger.bacc, losses=metric_logger.loss))

        else:
            print('* F1 {top1.global_avg:.4f} ACC {top5.global_avg:.4f} loss {losses.global_avg:.4f}'
                .format(top1=metric_logger.f1, top5=metric_logger.acc, losses=metric_logger.loss))
    
    metrics = get_topk_stats(similarities, k)

    return similarities, return_embeddings, metrics

@torch.no_grad()
def model_test(model_name):    
    checkpoint = torch.load(os.path.join(models_root_path, model_name, "best_checkpoint.pth"), map_location='cpu')
    args = checkpoint["args"]
    device = "cpu"

    model = create_model(
        args.model,
        pretrained=False,
        num_classes=args.nb_classes, #1000 for pretrained
        drop_rate=args.drop,
        drop_path_rate=args.drop_path,
        drop_block_rate=None,
        img_size=args.input_size
    )

    # model.load_state_dict(checkpoint['model_ema'])
    model.load_state_dict(checkpoint['model'])
    dataset_val, similarity_dataset, _ = build_dataset(is_train=False, is_test=True, args=args)
    sampler_val = torch.utils.data.SequentialSampler(dataset_val)
    data_loader = torch.utils.data.DataLoader(
            dataset_val, sampler=sampler_val,
            batch_size=int(1.5 * args.batch_size),
            num_workers=args.num_workers,
            pin_memory=args.pin_mem,
            drop_last=False
        )
    if args.bce_loss:
        criterion = torch.nn.BCEWithLogitsLoss()
    else:
        criterion = torch.nn.CrossEntropyLoss()

    metric_logger = utils.MetricLogger(delimiter="  ")
    header = 'Model Test:'

    # switch to evaluation mode
    model.eval()

    for batch in metric_logger.log_every(data_loader, 10, header):
        if args.data_set == 'WSS':
            images = batch["image"]
            target = batch["label"]
        
        if args.data_set == 'patch_cam' or args.data_set == 'mhist' or args.data_set == 'crc':
            images, target = batch
        
        images = images.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)

        # compute output
        with torch.cuda.amp.autocast():
            output = model(images)
            if args.data_set == 'crc':
                output = output
            else:
                output = output.squeeze(1) #not in crc
            loss = criterion(output, target)

        # f1, acc = accuracy(output, target, topk=(1, 5))
        if args.data_set == 'crc':
            pred = output.softmax(dim=1)
            _, pred = torch.max(pred, 1)
            pred = np.array(pred.detach().cpu(), dtype=int) #softmax!
        else:
            pred = output.sigmoid(dim=1)
            pred = np.array(output.detach().cpu() > 0.5, dtype=float)
        
        if args.data_set == 'WSS':
            f1 = f1_score(target.detach().cpu(), pred, average='samples')
        if args.data_set == 'crc':
            f1 = f1_score(y_true=np.array(target.detach().cpu()), y_pred=pred, average='weighted')
        if args.data_set == 'patch_cam' or args.data_set == 'mhist':
            f1 = f1_score(target.detach().cpu(), pred, average='binary')
        acc = accuracy_score(target.detach().cpu(), pred)
        if args.data_set == 'patch_cam' or args.data_set == 'mhist' or args.data_set == 'crc':
            bacc = balanced_accuracy_score(target.detach().cpu(), pred)

        batch_size = images.shape[0]
        metric_logger.update(loss=loss.item())
        metric_logger.meters['f1'].update(f1.item(), n=batch_size)
        metric_logger.meters['acc'].update(acc.item(), n=batch_size)
        if args.data_set == 'patch_cam' or args.data_set == 'mhist' or args.data_set == 'crc':
            metric_logger.meters['bacc'].update(bacc.item(), n=batch_size)
        
    # print(f"Model F1-Score: {f1.item()}")
    # print(f"Model acc: {acc.item()}")
    # if args.data_set == 'patch_cam' or args.data_set == 'mhist':
    #     print(f"Model bacc: {bacc.item()}")
    if args.data_set == 'patch_cam' or args.data_set == 'mhist' or args.data_set == 'crc':
        print('* F1 {top1.global_avg:.4f} ACC {top5.global_avg:.4f} BACC {top6.global_avg:.4f} loss {losses.global_avg:.4f}'
            .format(top1=metric_logger.f1, top5=metric_logger.acc, top6=metric_logger.bacc, losses=metric_logger.loss))
    else:
        print('* F1 {top1.global_avg:.4f} ACC {top5.global_avg:.4f} loss {losses.global_avg:.4f}'
            .format(top1=metric_logger.f1, top5=metric_logger.acc, losses=metric_logger.loss))
    
def distance(bob, binary_bob_embeddings):
        '''
        Function to compute the distance between two BoBs
        '''
        # Initialize the total distance
        total_dist = []
        for feat in binary_bob_embeddings:
            # Compute the distance to all barcodes in the other BoB
            distance = [butil.count_xor(bob[2], feat[2])]
            # Append the minimum distance
            total_dist.append([feat[0], feat[1], np.min(distance)])
            # total_dist.append(np.min(distances))
        # Return the median distance
        for item in total_dist:
            item[2] = np.median(item[2])
        
        retval = total_dist
        # Return the median distance
        return retval

def yottixel_search(embeddings, k, image_id):
    bob_embeddings = []
    binary_bob_embeddings = []
    for embedding in embeddings:
        bob_embedding = (np.diff(np.array(embedding[2]), axis=1) < 0)*1
        bob_embeddings.append([embedding[0], embedding[1], bob_embedding])

    for item in bob_embeddings:
        binary_bob_embeddings.append([item[0], item[1], bitarray.bitarray(item[2][0].tolist())])

    # Compute the distances
    # search_index = get_search_image_index(all_ids, image_id, False) #not needed in patchcam
    search_index = 0
    distances = [distance(binary_bob_embeddings[search_index], binary_bob_embeddings)]
    distances[0].sort(key = lambda row: row[2], reverse=False)
    get_topk_stats(distances[0], k)
    return distances[0]

def yottixel_slide_search(embeddings, k, image_id):
    bob_embeddings = []
    binary_bob_embeddings = []
    for embedding in embeddings:
        bob_embedding = (np.diff(np.array(embedding[2]), axis=1) < 0)*1
        bob_embeddings.append([embedding[0], embedding[1], bob_embedding])

    for item in bob_embeddings:
        binary_bob_embeddings.append([item[0], item[1], bitarray.bitarray(item[2][0].tolist())])

    # Compute the distances
    # search_index = get_search_image_index(all_ids, image_id, False) #not needed in patchcam
    search_index = 0
    distances = [distance(binary_bob_embeddings[search_index], binary_bob_embeddings)]
    distances_df = pd.DataFrame(distances[0], columns=("id", "target", "distance")) 
    distances_df = distances_df.groupby("id").sum()
    distances_df["patient_id"] = distances_df.index
    distances_df = distances_df[["patient_id", "target", "distance"]]

    distances_df["target"] = [target > torch.tensor([0,0,0]) for target in distances_df["target"]]
    distances_df["target"] = [[1 if label == torch.tensor(True) else 0 for label in target] for target in distances_df["target"]]

    distances_np = distances_df.to_numpy()
    distances_list = distances_np.tolist()
    
    # search_index = get_search_image_index(all_ids, image_id, False) #not needed in patchcam
    search_index = 0
    search_image = distances_list[search_index]
    distances_list.sort(key = lambda row: row[2], reverse=False)
    distances_list.remove(search_image)
    distances_list.insert(0, search_image)

    get_topk_stats(distances_list, k)
    return distances_list
    
def get_topk_stats(similarities, k):
    metrics = {}
    search_image = similarities[0]
    similarities = similarities[1:]
    relavent = 0.01
    total_relavent = 0.01
    topk_similarities = similarities[:k]
    dcg = 0.01
    idcg = 0.01

    for i in range(k):
        idcg = idcg + 1 / math.log2((i+1) + 1)

    for item in similarities:
        if (np.asarray(search_image[1]) == np.asarray(item[1])).all():
            total_relavent += 1

    for i, item in enumerate(topk_similarities):
        if (np.asarray(search_image[1]) == np.asarray(item[1])).all():
            relavent += 1
            dcg = dcg + 1/(math.log2((i+1)+1))
        else:
            dcg = dcg + 0/(math.log2((i+1)+1))
            
    percision = relavent/(len(topk_similarities))
    # recall = relavent/(total_relavent)
    recall = relavent/(k)
    ndcg = dcg/idcg
    similarity_f1_score = 2*((percision*recall)/(percision+recall))

    # print(f"Percision: {percision}, Recall: {recall}, NDCG: {ndcg}")
    # print(f"Similarity Test F1-Score: {similarity_f1_score}")

    metrics = {"percision" : percision, "recall" : recall, "NDCG" : ndcg, "f1" : similarity_f1_score}

    return metrics

def get_camelyon_embeds(model):
    path = f"../../../../../../../../mnt/data/nfs03-R6/CAMELYON17/additional_features/feature_vectors/pca/{model}"
    data = pd.read_csv("../../../../../../mnt/data/nfs03-R6/CAMELYON17/additional_features/slides.csv", index_col=0)
    regions_pca = PCA(n_components=1)
    samples = next(os.walk(path))[2]
    embeds = [[],[],[]]

    for i, sample in enumerate(samples):
        if i%100 == 0:
            print(i)
        with h5py.File(os.path.join(path,sample), mode='r') as file:
            embed = file['feature_vector'][:]

        #Apply PCA
        regions_pca_output = regions_pca.fit_transform(embed.T)
        pca_output = regions_pca_output.T
        embeds[0].append(f"{sample[:-3]}.tif")
        embeds[1].append(data.loc[data['patient'] == f"{sample[:-3]}.tif"]["stage"])
        embeds[2].append(pca_output)

    embeds_tensors = [embeds[0], embeds[1], torch.tensor(embeds[2])]

    return embeds_tensors  

# Loading Model

In [13]:
# model_name = "camelyon_16_pca_general_pretrained_deit_tiny_patch16_224"
model_name = "camelyon_pca_pretrained_mocov3_tiny_checkpoint"

In [14]:
checkpoint = torch.load(os.path.join(models_root_path, model_name, "best_checkpoint.pth"), map_location='cpu')
args = checkpoint["args"]
checkpoint["epoch"]

7

In [12]:
checkpoint["model_out"]

OrderedDict([('norm.weight',
              tensor([0.9970, 0.9946, 0.9987, 0.9948, 0.9945, 1.0051, 0.9943, 1.0016, 1.0039,
                      1.0017, 0.9993, 0.9991, 1.0023, 0.9934, 1.0061, 0.9972, 0.9943, 1.0012,
                      1.0019, 1.0034, 1.0056, 0.9964, 1.0029, 1.0012, 1.0032, 0.9966, 0.9970,
                      1.0019, 1.0027, 0.9963, 1.0042, 0.9939, 1.0150, 1.0037, 1.0054, 1.0010,
                      0.9873, 0.9961, 0.9958, 0.9954, 0.9906, 0.9956, 0.9987, 0.9993, 0.9956,
                      1.0028, 0.9944, 1.0023, 1.0059, 0.9998, 1.0095, 1.0047, 1.0028, 1.0029,
                      0.9910, 0.9932, 1.0038, 0.9980, 1.0041, 0.9962, 0.9963, 0.9964, 0.9950,
                      0.9940, 0.9953, 0.9979, 1.0040, 1.0010, 1.0036, 0.9998, 1.0021, 1.0077,
                      0.9953, 1.0130, 0.9859, 0.9956, 0.9909, 0.9967, 1.0016, 0.9982, 0.9943,
                      0.9977, 1.0050, 0.9886, 0.9979, 1.0024, 0.9917, 1.0055, 1.0031, 0.9907,
                      0.9983, 1

In [11]:
from models.deit.model_attn_wrapper import AttentionMIL
model = AttentionMIL(input_dim = 192, num_classes=1)
model

AttentionMIL(
  (encoder): Sequential(
    (0): Linear(in_features=192, out_features=100, bias=True)
    (1): ReLU()
  )
  (attention): MILAttention(
    (linear1): Linear(in_features=100, out_features=50, bias=True)
    (tanh): Tanh()
    (linear2): Linear(in_features=50, out_features=1, bias=True)
  )
  (head): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=192, out_features=1, bias=True)
  )
)

In [16]:
model = create_model(
        args.model,
        pretrained=False,
        num_classes=args.nb_classes, #1000 for pretrained
        drop_rate=args.drop,
        drop_path_rate=args.drop_path,
        drop_block_rate=None,
        img_size=args.input_size
    )

# model.load_state_dict(checkpoint['model'])

In [17]:
model

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
    (1): Block(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine

In [ ]:
sum(p.numel() for p in model.parameters())

In [ ]:
sum(p.numel() for p in model.parameters())

In [ ]:
sum(p.numel() for p in model.parameters())

In [ ]:
checkpoint.keys()

# Load Sample

In [ ]:
image_id = "31" #similar: 33 opposite:02
wss_features = pd.read_csv(os.path.join(data_root_path, "WSSS4LUAD", "features/validation.csv"), index_col=0)
wss_features['id'] = wss_features['id'].astype(str).str.zfill(2)
image = io.imread(os.path.join(data_root_path, "WSSS4LUAD", f"2.validation/img/{image_id}.png"))
label = wss_features[wss_features.id == image_id]["patch_level_multi_class_labels"].item()
print("ID:", image_id, "Label:", label)

In [ ]:
io.imshow(image)

In [ ]:
val_transforms  = Compose([EnsureChannelFirst(channel_dim=2), ScaleIntensityRange(a_min=0.0, a_max=2.0, b_min=0.0, b_max=1.0), torchvision.transforms.Resize((224,224))])
transformed_image = val_transforms(image)
transformed_image = transformed_image[None, :, :, :]
transformed_image.shape

# Inference and Embedding

In [ ]:
outputs = model(transformed_image)
predictions = torch.round(torch.nn.Softmax()(outputs))
predictions

In [ ]:
model

In [ ]:
activation = {}
def get_activation(name):
    def hook(model, input, output):
        activation[name] = output.detach()
    return hook

model.pre_logits.register_forward_hook(get_activation("embed"))
intermediate_outputs = model(transformed_image)
print(activation['embed'].flatten().reshape(1, -1).shape)

# Model evaluation

In [ ]:
# model_name = "deit_tiny_patch16_224"
# model_name = "general_pretrained_deit_tiny_patch16_224" #best in WSS classification
# model_name = "pretrained_mocov3_tiny_checkpoint"

# model_name = "patch_cam_deit_tiny_patch16_224"
# model_name = "patch_cam_general_pretrained_deit_tiny_patch16_224" #best in patchcam classification
# model_name = "patch_cam_pretrained_mocov3_tiny_checkpoint"

# model_name = "mhist_deit_tiny_patch16_224" 
# model_name = "mhist_general_pretrained_deit_tiny_patch16_224" 
# model_name = "mhist_pretrained_mocov3_tiny_checkpoint" #best in mhist classification

# model_name = "crc_deit_tiny_patch16_224" 
# model_name = "crc_general_pretrained_deit_tiny_patch16_224" 
model_name = "crc_pretrained_mocov3_tiny_checkpoint" #best in crc classification

In [ ]:
model_test(model_name)

# Calculating Similarity

In [ ]:
# model_name = "deit_tiny_patch16_224"
# resume = "https://dl.fbaipublicfiles.com/deit/deit_tiny_patch16_224-a1311bcf.pth"
# model_name = "deit_small_patch16_224"
# resume = "../../../../../../../../projects/fm_histopathology/models/mocov3_tiny_checkpoint/mocov3_deit_vit_small.pth"

# model_name = "wss_deit_tiny_patch16_224"
# model_name = "wss_general_pretrained_deit_tiny_patch16_224" #best in WSS classification
# model_name = "wss_pretrained_mocov3_tiny_checkpoint"

# model_name = "patch_cam_deit_tiny_patch16_224"
# model_name = "patch_cam_general_pretrained_deit_tiny_patch16_224" #best in patchcam classification
# model_name = "patch_cam_pretrained_mocov3_tiny_checkpoint"

# model_name = "mhist_deit_tiny_patch16_224" 
# model_name = "mhist_general_pretrained_deit_tiny_patch16_224" 
# model_name = "mhist_pretrained_mocov3_tiny_checkpoint" #best in mhist classification

# model_name = "crc_deit_tiny_patch16_224" 
# model_name = "crc_general_pretrained_deit_tiny_patch16_224" 
# model_name = "crc_pretrained_mocov3_tiny_checkpoint" #best in crc classification

#WSI
# model_name = "camelyon_pca_deit_tiny_patch16_224"  #best in camelyon classification
# model_name = "camelyon_pca_general_pretrained_deit_tiny_patch16_224" 
# model_name = "camelyon_pca_pretrained_mocov3_tiny_checkpoint" #best in camelyon classification
model_name = "camelyon_UNI"
embeds = get_camelyon_embeds("UNI")


In [ ]:
similarities, embeddings, _ = similarity_test(model_name, 5, 0, embeds) #1005919, 31,6,0, 1226
similarities

In [ ]:
similarities, embeddings, _ = similarity_test(model_name, 5, 0, None, resume, "crc") #1005919, 31,6,0, 1226
similarities

In [ ]:
total_metrics = {"percision" : 0.0, "recall" : 0.0, "NDCG" : 0.0, "f1" : 0.0}
count = len(embeddings)
print(count)
for i in range(0, count):
    if i%100 == 0:
        print(i)
    if not model_name[:8] == "camelyon":
        _, _, metrics = similarity_test(model_name, 5, i, None, "yes", "patch_cam")
    else: 
        _, _, metrics = similarity_test(model_name, 5, i, embeds)
    total_metrics["percision"] = total_metrics["percision"] + metrics["percision"]
    total_metrics["recall"] = total_metrics["recall"] + metrics["recall"]
    total_metrics["NDCG"] = total_metrics["NDCG"] + metrics["NDCG"]
    total_metrics["f1"] = total_metrics["f1"] + metrics["f1"]

print({total_metrics['f1']/count}, {total_metrics['percision']/count}, 
      {total_metrics['recall']/count}, {total_metrics['NDCG']/count})

In [ ]:
for i, embedding in enumerate(embeddings):
    if embedding[0] == 2.0:
        print(i)
        break

In [ ]:
similarities = yottixel_search(embeddings, 5, 1005919)  #1005919
similarities

### Distribution

In [ ]:
labels = []
for item in embeddings:
    labels.append(item[1])

labels = np.asanyarray(labels)
labels = [str(label) for label in labels]
pd.DataFrame(labels).value_counts()

In [ ]:
training_features = pd.read_csv("../../../../../mnt/data/WSSS4LUAD/features/training.csv", index_col=0)
training_features.patch_level_multi_class_labels.value_counts()

In [ ]:
x = pd.read_csv("../../../../../mnt/data/WSSS4LUAD/features/validation.csv", index_col=0)
x[x['patch_level_multi_class_labels'].isin(["[1, 1, 1]"])]
# x[x['id'].isin(["1021408"])]

### Slide Level retrival

In [ ]:
similarities = yottixel_slide_search(embeddings, 5, 1005919)
similarities

In [ ]:
df = pd.read_csv("../../../../../mnt/data/WSSS4LUAD/features/validation.csv", index_col=0)
df['id'] = df['id'].astype(str).str.zfill(2)
df

In [ ]:
x = pd.read_csv("../../../../../mnt/data/WSSS4LUAD/features/validation.csv", index_col=0)
x[x['patch_level_multi_class_labels'].isin(["[1, 1, 1]"])]
# x[x['id'].isin(["1021408"])]

In [ ]:
m = torch.nn.Softmax(dim=0)
x = torch.randn(1)
print(x)
output = m(x)
print(output)

In [ ]:
model_name = "mhist_deit_tiny_patch16_224" 
# model_name = "mhist_pretrained_mocov3_tiny_checkpoint" 
checkpoint = torch.load(os.path.join(models_root_path, model_name, "best_checkpoint.pth"), map_location='cpu')
device = "cpu"
args = checkpoint["args"]
model = create_model(
            args.model,
            pretrained=False,
            num_classes=args.nb_classes, #1000 for pretrained
            drop_rate=args.drop,
            drop_path_rate=args.drop_path,
            drop_block_rate=None,
            img_size=args.input_size
        )

model.load_state_dict(checkpoint['model'])

In [ ]:
def get_activation(name):
    def hook(model, input, output):
        activation[name] = output.detach()
    return hook

In [ ]:
model.norm.normalized_shape[0]

In [ ]:
list(model.children())[-4]

In [ ]:
model.pre_logits

In [ ]:
# Print weights after loading the checkpoint
print("\nWeights of model_core after loading checkpoint:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"Layer: {name} | Weights: {param.data}")


In [ ]:
layers = list(model.children())

# Remove the last three layers
core_layers = layers[:-3]

# Rebuild the model without the last three layers
model_core = nn.Sequential(*core_layers)

# # Print weights after loading the checkpoint
# print("\nWeights of model_core after loading checkpoint:")
# for name, param in model_core.named_parameters():
#     if param.requires_grad:
#         print(f"Layer: {name} | Weights: {param.data}")
model_core


In [ ]:
model_core[-1][-1].mlp.fc2.out_features

In [ ]:
last_three_layers = layers[-3:]

# Rebuild the model without the last three layers
model_out = nn.Sequential(*last_three_layers)
model_out

In [ ]:
from sklearn.decomposition import PCA
x1 = torch.randn(10, 12)


pca = PCA(n_components=1)
pca_output = pca.fit_transform(x1.T)
pca_output.shape

# End